# Stock Price Prediction — 6 Models Comparison
### Stocks: AAPL, NVDA, MSFT, GOOGL
### Models: LSTM, Linear Regression, Random Forest, SVR, GRU, XGBoost

In [ ]:
# Install XGBoost if not installed
!pip install xgboost -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU

import xgboost as xgb

print('All libraries loaded successfully!')

In [ ]:
stocks     = ['AAPL', 'NVDA', 'MSFT', 'GOOGL']
time_step  = 60
all_results = []

## Loop through each stock and train all 6 models

In [ ]:
def get_metrics(real, pred, mean_price):
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    r2   = r2_score(real, pred)
    acc  = 100 - (mae / mean_price * 100)
    return round(mae,2), round(rmse,2), round(r2,4), round(acc,2)


for ticker in stocks:

    print(f'\n==============================')
    print(f'Processing: {ticker}')
    print(f'==============================')

    # ---------- Data ----------
    df = yf.download(ticker, start='2015-01-01', end='2023-12-31')
    df = df[['Close']].dropna()

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)

    X, y = [], []
    for i in range(time_step, len(scaled)):
        X.append(scaled[i-time_step:i, 0])
        y.append(scaled[i, 0])

    X = np.array(X)
    y = np.array(y)

    split     = int(len(X) * 0.8)
    X_train   = X[:split]
    X_test    = X[split:]
    y_train   = y[:split]
    y_test    = y[split:]

    mean_price = scaler.inverse_transform(y_test.reshape(-1,1)).mean()

    # ========== 1. LSTM ==========
    print('Training LSTM...')
    X_lstm_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_lstm_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    lstm = Sequential([
        LSTM(50, return_sequences=True, input_shape=(time_step, 1)),
        LSTM(50),
        Dense(1)
    ])
    lstm.compile(optimizer='adam', loss='mse')
    lstm.fit(X_lstm_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred_lstm = scaler.inverse_transform(lstm.predict(X_lstm_test, verbose=0))
    real      = scaler.inverse_transform(y_test.reshape(-1,1))
    mae, rmse, r2, acc = get_metrics(real, pred_lstm, mean_price)
    all_results.append([ticker, 'LSTM', mae, rmse, r2, acc])
    print(f'  LSTM Accuracy: {acc}%')

    # ========== 2. GRU ==========
    print('Training GRU...')
    gru = Sequential([
        GRU(50, return_sequences=True, input_shape=(time_step, 1)),
        GRU(50),
        Dense(1)
    ])
    gru.compile(optimizer='adam', loss='mse')
    gru.fit(X_lstm_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred_gru = scaler.inverse_transform(gru.predict(X_lstm_test, verbose=0))
    mae, rmse, r2, acc = get_metrics(real, pred_gru, mean_price)
    all_results.append([ticker, 'GRU', mae, rmse, r2, acc])
    print(f'  GRU Accuracy: {acc}%')

    # ========== 3. Linear Regression ==========
    print('Training Linear Regression...')
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    pred_lr = scaler.inverse_transform(lr.predict(X_test).reshape(-1,1))
    mae, rmse, r2, acc = get_metrics(real, pred_lr, mean_price)
    all_results.append([ticker, 'Linear Regression', mae, rmse, r2, acc])
    print(f'  Linear Regression Accuracy: {acc}%')

    # ========== 4. Random Forest ==========
    print('Training Random Forest...')
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    pred_rf = scaler.inverse_transform(rf.predict(X_test).reshape(-1,1))
    mae, rmse, r2, acc = get_metrics(real, pred_rf, mean_price)
    all_results.append([ticker, 'Random Forest', mae, rmse, r2, acc])
    print(f'  Random Forest Accuracy: {acc}%')

    # ========== 5. SVR ==========
    print('Training SVR...')
    svr = SVR(kernel='rbf', C=100, epsilon=0.01)
    svr.fit(X_train, y_train)
    pred_svr = scaler.inverse_transform(svr.predict(X_test).reshape(-1,1))
    mae, rmse, r2, acc = get_metrics(real, pred_svr, mean_price)
    all_results.append([ticker, 'SVR', mae, rmse, r2, acc])
    print(f'  SVR Accuracy: {acc}%')

    # ========== 6. XGBoost ==========
    print('Training XGBoost...')
    xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    pred_xgb = scaler.inverse_transform(xgb_model.predict(X_test).reshape(-1,1))
    mae, rmse, r2, acc = get_metrics(real, pred_xgb, mean_price)
    all_results.append([ticker, 'XGBoost', mae, rmse, r2, acc])
    print(f'  XGBoost Accuracy: {acc}%')

print('\nAll models done!')

## Results Table — All Models vs All Stocks

In [ ]:
results_df = pd.DataFrame(
    all_results,
    columns=['Stock', 'Model', 'MAE', 'RMSE', 'R2', 'Accuracy %']
)

results_df

## Best Model per Stock

In [ ]:
best = results_df.loc[results_df.groupby('Stock')['Accuracy %'].idxmax()]
best

## Chart — Accuracy Comparison per Stock

In [ ]:
models = results_df['Model'].unique()
x      = np.arange(len(stocks))
width  = 0.13

fig, ax = plt.subplots(figsize=(14, 6))

for i, model in enumerate(models):
    vals = results_df[results_df['Model'] == model]['Accuracy %'].values
    ax.bar(x + i * width, vals, width, label=model)

ax.set_xlabel('Stock')
ax.set_ylabel('Accuracy %')
ax.set_title('Model Accuracy Comparison per Stock')
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(stocks)
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## Summary

This notebook compares 6 models on 4 stocks:

- **LSTM** — Long Short-Term Memory (original model)
- **GRU** — Gated Recurrent Unit (similar to LSTM but simpler)
- **Linear Regression** — simple baseline model
- **Random Forest** — ensemble tree-based model
- **SVR** — Support Vector Regression
- **XGBoost** — powerful gradient boosting model

Accuracy formula: `100 - (MAE / mean_price * 100)`